In [ ]:
"""
----
같은 조합 코드의 이미지가 train/val에 나뉘어 들어가지 않도록 조합 코드 단위로 그룹핑한 stratified split을 생성한다.


- 조합 코드 단위로 그룹핑 -> 조합 하나가 통째로 train 또는 val 중 하나에만 배정
- MultilabelStratifiedShuffleSplit(iterative-stratification 라이브러리)으로
  118개 클래스 분포를 최대한 균형있게 유지하며 8:2 분할

"""

In [ ]:
import os
import json
import shutil
from collections import defaultdict
import yaml

import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

BASE_PATH = "./data/dataset_combined_final"
CLASS_MAP_PATH = "./data/class_map_118.json"
EXCLUDED_JSON = "./data/excluded_problem_images_v4final.json"

OUTPUT_DIR = "./data/dataset_stratified_v4"
NUM_CLASSES = 118
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [ ]:
def extract_combo_code(fname: str) -> str:
    parts = fname.replace(".png", "").replace(".txt", "").split("_")
    for p in parts:
        if p.startswith("K-"):
            return p
    return fname

In [ ]:
def load_all_labels():
    img_id_to_boxes = {}
    img_id_to_fname = {}
    img_id_to_split_orig = {}
    for split in ("train", "val"):
        img_dir = f"{BASE_PATH}/images/{split}"
        label_dir = f"{BASE_PATH}/labels/{split}"
        for fname in os.listdir(img_dir):
            img_id = fname.replace(".png", "")
            img_id_to_fname[img_id] = fname
            img_id_to_split_orig[img_id] = split
        for fname in os.listdir(label_dir):
            img_id = fname.replace(".txt", "")
            boxes = []
            with open(f"{label_dir}/{fname}") as f:
                for line in f:
                    parts = line.split()
                    cls_id = int(parts[0])
                    boxes.append(cls_id)
            img_id_to_boxes[img_id] = boxes
    return img_id_to_boxes, img_id_to_fname, img_id_to_split_orig

In [ ]:
with open(CLASS_MAP_PATH, encoding="utf-8") as f:
    class_map = json.load(f)
with open(EXCLUDED_JSON, encoding="utf-8") as f:
    excluded_ids = set(json.load(f))

img_id_to_boxes, img_id_to_fname, img_id_to_split_orig = load_all_labels()

valid_ids = [img_id for img_id in img_id_to_boxes if img_id not in excluded_ids]
print(f"정제 전: {len(img_id_to_boxes)} -> 정제 후: {len(valid_ids)}")

combo_to_images = defaultdict(list)
combo_to_classes = defaultdict(set)
for img_id in valid_ids:
    fname = img_id_to_fname[img_id]
    combo = extract_combo_code(fname)
    combo_to_images[combo].append(img_id)
    combo_to_classes[combo].update(img_id_to_boxes[img_id])

combo_ids = list(combo_to_images.keys())
print(f"고유 조합 코드 수: {len(combo_ids)}")

y_combo = np.zeros((len(combo_ids), NUM_CLASSES), dtype=int)
for row_idx, combo in enumerate(combo_ids):
    for cat in combo_to_classes[combo]:
        y_combo[row_idx, cat] = 1
X_combo = np.array(combo_ids)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_combo_idx, val_combo_idx = next(msss.split(X_combo, y_combo))
train_combos = set(X_combo[train_combo_idx])
val_combos = set(X_combo[val_combo_idx])

train_ids = [img_id for combo in train_combos for img_id in combo_to_images[combo]]
val_ids = [img_id for combo in val_combos for img_id in combo_to_images[combo]]
print(f"최종 split -> train: {len(train_ids)} / val: {len(val_ids)}")

assert len(train_combos & val_combos) == 0, "조합 코드가 train/val에 동시에 존재함"
val_class_coverage = set()
for img_id in val_ids:
    val_class_coverage.update(img_id_to_boxes[img_id])
print(f"val 클래스 커버리지: {len(val_class_coverage)}/{NUM_CLASSES}")

for split in ("train", "val"):
    os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

for img_id, split in [(i, "train") for i in train_ids] + [(i, "val") for i in val_ids]:
    fname = img_id_to_fname[img_id]
    orig_split = img_id_to_split_orig[img_id]
    src_img = f"{BASE_PATH}/images/{orig_split}/{fname}"
    src_lbl = f"{BASE_PATH}/labels/{orig_split}/{img_id}.txt"
    shutil.copy(src_img, f"{OUTPUT_DIR}/images/{split}/{fname}")
    shutil.copy(src_lbl, f"{OUTPUT_DIR}/labels/{split}/{img_id}.txt")

data_yaml = {
    "path": os.path.abspath(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": NUM_CLASSES,
    "names": [class_map["idx_to_name"][str(i)] for i in range(NUM_CLASSES)],
}
with open(f"{OUTPUT_DIR}/data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

with open("./data/stratified_train_ids_v4final.json", "w") as f:
    json.dump(train_ids, f)
with open("./data/stratified_val_ids_v4final.json", "w") as f:
    json.dump(val_ids, f)

print(f"\ndataset_stratified_v4 생성 완료: {OUTPUT_DIR}")